# Week 7 Assignment

## Delta Lake MERGE Implementation

### Objective

Perform incremental data processing using Delta Lake by loading the Superstore dataset, cleaning it, creating incremental records, applying MERGE, and validating the results.

In [0]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/week7data/Sample - Superstore.csv")

display(df)

In [0]:
df.printSchema()

In [0]:
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

## Loading Dataset into Delta Table

In this step, the Superstore dataset is stored in Delta format for performing incremental data processing.

In [0]:
from pyspark.sql.functions import col

# Replace spaces with underscores
df = df.toDF(*[c.replace(" ", "_") for c in df.columns])

# Check new column names
print(df.columns)

In [0]:
df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("superstore_delta")

In [0]:
delta_df = spark.table("superstore_delta")

display(delta_df)

## Data Cleaning

In this step, null values and duplicate records are removed from the dataset to improve data quality before performing the MERGE operation.

In [0]:
from pyspark.sql.functions import col, when, count

delta_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in delta_df.columns
]).display()

In [0]:
clean_df = delta_df.na.drop()

In [0]:
clean_df = clean_df.dropDuplicates()

In [0]:
display(clean_df)

In [0]:
clean_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("superstore_delta")

## Create Incremental Dataset

An incremental dataset is created to simulate new incoming data. It contains:
- Existing records (to be updated)
- New records (to be inserted)

In [0]:
clean_df = spark.table("superstore_delta")
display(clean_df)

In [0]:
from pyspark.sql.functions import col

clean_df = clean_df \
.withColumn("Sales", col("Sales").cast("double")) \
.withColumn("Quantity", col("Quantity").cast("int")) \
.withColumn("Discount", col("Discount").cast("double")) \
.withColumn("Profit", col("Profit").cast("double"))

clean_df.printSchema()

In [0]:
from pyspark.sql.functions import col

existing = clean_df.filter(col("Row_ID").isin([1,2]))
existing = existing.withColumn("Sales", col("Sales") + 100)

new_records = clean_df.filter(col("Row_ID").isin([3,4]))
new_records = new_records.withColumn("Row_ID", col("Row_ID") + 10000)

incremental_df = existing.union(new_records)

display(incremental_df)

In [0]:
from delta.tables import DeltaTable

In [0]:
deltaTable = DeltaTable.forName(
    spark,
    "superstore_delta"
)

# Applying Merge Operation

In [0]:
deltaTable.alias("target") \
.merge(
    incremental_df.alias("source"),
    "target.Row_ID = source.Row_ID"
) \
.whenMatchedUpdate(
    set={
        "Sales": "source.Sales"
    }
) \
.whenNotMatchedInsertAll() \
.execute()

In [0]:
final_df = spark.table("superstore_delta")

display(final_df)

In [0]:
duplicates = final_df.groupBy("Row_ID") \
    .count() \
    .filter("count > 1")

display(duplicates)

In [0]:
final_df.describe().show()

# Conclusion

Successfully implemented Delta Lake MERGE using the Superstore dataset.

### Tasks Completed

- Loaded CSV dataset into a Delta Table.
- Performed data cleaning by removing null values and duplicate records.
- Created an incremental dataset to simulate new incoming data.
- Updated existing records and inserted new records using Delta Lake MERGE.
- Validated the row count and confirmed successful updates and inserts.
- Verified that there were no duplicate records after the MERGE operation.
- Displayed the final dataset and summary statistics.

This assignment demonstrates how Delta Lake MERGE enables efficient UPSERT operations for incremental data processing in modern data engineering workflows.